<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="300" alt="Skills Network Logo">
    </a>
</p>


# Practice Project: GDP Data extraction and processing

Estimated time needed: **30** minutes

## Introduction

In this practice project, you will put the skills acquired through the course to use. You will extract data from a website using webscraping and reqeust APIs process it using Pandas and Numpy libraries.


## Project Scenario:

An international firm that is looking to expand its business in different countries across the world has recruited you. You have been hired as a junior Data Engineer and are tasked with creating a script that can extract the list of the top 10 largest economies of the world in descending order of their GDPs in Billion USD (rounded to 2 decimal places), as logged by the International Monetary Fund (IMF). 

The required data seems to be available on the URL mentioned below:


URL: https://web.archive.org/web/20230902185326/https://en.wikipedia.org/wiki/List_of_countries_by_GDP_%28nominal%29


## Objectives

After completing this lab you will be able to:

 - Use Pandas to read the data from html page and process the tabular data as a dataframe.
 - Use Numpy to manipulate the information contatined in the dataframe.
 - Load the updated dataframe to CSV file.


---


## Dislcaimer

If you are using a downloaded version of this notebook on your local machine, you may encounter a warning message as shown in the screenshot below.

<p style="text-align:center">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0101EN-SkillsNetwork/labs/mod_5/practice_project_disclaimer.png" width="700" alt="warning message">
</p>


This does not affect the execution of your codes in any way and can be simply ignored. 


# Setup


For this lab, we will be using the following libraries:

*   [`pandas`](https://pandas.pydata.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) for managing the data.
*   [`numpy`](https://numpy.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) for mathematical operations.


### Importing Required Libraries

_We recommend you import all required libraries in one place (here):_


In [35]:
import numpy as np
import pandas as pd

# You can also use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

---


# Exercises

### Exercise 1
Extract the required GDP data from the given URL using Web Scraping.


In [2]:
URL="https://web.archive.org/web/20230902185326/https://en.wikipedia.org/wiki/List_of_countries_by_GDP_%28nominal%29"

You can use Pandas library to extract the required table directly as a DataFrame. Note that the required table is the third one on the website, as shown in the image below.

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0101EN-SkillsNetwork/images/pandas_wbs_3.png">


**Solution**

---

**1. Extracting the table using Beautiful Soup**

In [8]:
# Import necessary libraries
from bs4 import BeautifulSoup
import requests

# Get contents of the URL in text format and store it to the variable "data"
data = requests.get(URL).text
print(requests.get(URL).headers['Content-Type'])

text/html; charset=UTF-8


Contents of the URL has been stored in `'data'` and it is clear that it is an HTML document. `BeautifulSoup` can then be used to parse the contents by passing `'data'` into the `BeautifulSoup` constructor.

In [9]:
data_soup = BeautifulSoup(data, 'html5lib')

The tables can then be extracted using the `find` or `find_all` methods.

In [10]:
data_tables = data_soup.find_all('table')
table = data_tables[2] #Since the exercise specifically asks to only extract Table 3

Table has been extracted but still as an HTML tag. This can still be used to extract the header and the rows. From the table, Country is the first column while the GDP forcasted by IMF is the third column. From this, a dictionary can be created containing a list of the top ten countries and their corresonding GDPs.

In [11]:
rows = {'Country':[],'GDP (Million USD)':[]}
for row in table.find_all('tr')[3:13]:
    country = row.find_all('a')[0].string
    rows['Country'].append(country)

    gdp = row.find_all('td')[2].text
    rows['GDP (Million USD)'].append(gdp)
rows

{'Country': ['United States',
  'China',
  'Japan',
  'Germany',
  'India',
  'United Kingdom',
  'France',
  'Italy',
  'Canada',
  'Brazil'],
 'GDP (Million USD)': ['26,854,599',
  '19,373,586',
  '4,409,738',
  '4,308,854',
  '3,736,882',
  '3,158,938',
  '2,923,489',
  '2,169,745',
  '2,089,672',
  '2,081,235']}

**2. Creating the Pandas dataframe**

---

Now to turn the dictionary to a Pandas dataframe.

In [30]:
import pandas as pd
gdp_dataframe = pd.DataFrame(rows)
gdp_dataframe

,Country,GDP (Million USD)
0,United States,"26,854,599"
1,China,"19,373,586"
2,Japan,"4,409,738"
3,Germany,"4,308,854"
4,India,"3,736,882"
5,United Kingdom,"3,158,938"
6,France,"2,923,489"
7,Italy,"2,169,745"
8,Canada,"2,089,672"
9,Brazil,"2,081,235"


<details>
    <summary>Click here for Solution</summary>

```python
# Extract tables from webpage using Pandas. Retain table number 3 as the required dataframe.
tables = pd.read_html(URL)
df = tables[3]

# Replace the column headers with column numbers
df.columns = range(df.shape[1])

# Retain columns with index 0 and 2 (name of country and value of GDP quoted by IMF)
df = df[[0,2]]

# Retain the Rows with index 1 to 10, indicating the top 10 economies of the world.
df = df.iloc[1:11,:]

# Assign column names as "Country" and "GDP (Million USD)"
df.columns = ['Country','GDP (Million USD)']
```

</details>


### Exercise 2
Modify the GDP column of the DataFrame, converting the value available in Million USD to Billion USD. Use the `round()` method of Numpy library to round the value to 2 decimal places. Modify the header of the DataFrame to `GDP (Billion USD)`.


**Solution.**

---

**1. Convert GDP to integers**

The GDP values stored in the dataframe are strings with commas. To be able to do mathematical operations on them, they first have to be converted to integers. Start by removing the commas using the `replace()` method.


In [31]:
gdp_dataframe['GDP (Million USD)'] = gdp_dataframe['GDP (Million USD)'].str.replace(',','')
gdp_dataframe

,Country,GDP (Million USD)
0,United States,26854599
1,China,19373586
2,Japan,4409738
3,Germany,4308854
4,India,3736882
5,United Kingdom,3158938
6,France,2923489
7,Italy,2169745
8,Canada,2089672
9,Brazil,2081235


Then, convert GDP to integers using the `astype()` method

In [ ]:
gdp_dataframe['GDP (Million USD)'] = gdp_dataframe['GDP (Million USD)'].astype(int)
gdp_dataframe['GDP (Million USD)'].dtypes

dtype('int64')

**2. Converting to Billion USD**

Mathematical operations can now be done on the GDP column. To convert from Billion USD to Million USD, simply divide the Million USD value by 1000.

In [33]:
gdp_dataframe['GDP (Million USD)'] = gdp_dataframe['GDP (Million USD)']/1000
gdp_dataframe

,Country,GDP (Million USD)
0,United States,26854.599
1,China,19373.586
2,Japan,4409.738
3,Germany,4308.854
4,India,3736.882
5,United Kingdom,3158.938
6,France,2923.489
7,Italy,2169.745
8,Canada,2089.672
9,Brazil,2081.235


Next, round the converted values to two decimal places using the numpy `round()` method.

In [ ]:
gdp_dataframe['GDP (Million USD)'] = np.round(gdp_dataframe['GDP (Million USD)'],2)

In [ ]:
gdp_dataframe

,Country,GDP (Million USD)
0,United States,26854.60
1,China,19373.59
2,Japan,4409.74
3,Germany,4308.85
4,India,3736.88
5,United Kingdom,3158.94
6,France,2923.49
7,Italy,2169.74
8,Canada,2089.67
9,Brazil,2081.24


**3. Renaming the Column**

Finally, replace the column name to 'GDP (Billion USD)' using the pandas `rename()` method.

In [40]:
gdp_dataframe = gdp_dataframe.rename(columns={"GDP (Million USD)":"GDP (Billion USD)"})
gdp_dataframe

,Country,GDP (Billion USD)
0,United States,26854.60
1,China,19373.59
2,Japan,4409.74
3,Germany,4308.85
4,India,3736.88
5,United Kingdom,3158.94
6,France,2923.49
7,Italy,2169.74
8,Canada,2089.67
9,Brazil,2081.24


<details>
    <summary>Click here for solution</summary>
    
```python
# Change the data type of the 'GDP (Million USD)' column to integer. Use astype() method.
df['GDP (Million USD)'] = df['GDP (Million USD)'].astype(int)

# Convert the GDP value in Million USD to Billion USD
df[['GDP (Million USD)']] = df[['GDP (Million USD)']]/1000

# Use numpy.round() method to round the value to 2 decimal places.
df[['GDP (Million USD)']] = np.round(df[['GDP (Million USD)']], 2)

# Rename the column header from 'GDP (Million USD)' to 'GDP (Billion USD)'
df.rename(columns = {'GDP (Million USD)' : 'GDP (Billion USD)'})

```
</details>


### Exercise 3


Load the DataFrame to the CSV file named "Largest_economies.csv"


**Solution**

---

**1. Defining the file path**

Start by defining the file path. Save the CSV file to current working directory using the `os.path.join()` method

In [41]:
import os
path = os.path.join(os.getcwd(),'Largest_economies.csv')
path

'/Users/jelo/Desktop/IBM-Data-Science-Specialization/Course-4_Python-Basics/Module-5/Largest_economies.csv'

**2. Dataframe to CSV**

We then load the dataframe to the CSV file using the pandas `DataFrame.to_csv()` method

In [42]:
gdp_dataframe.to_csv(path,index=False)

<details>
    <summary>Click here for Solution</summary>

```python
# Load the DataFrame to the CSV file named "Largest_economies.csv"
df.to_csv('./Largest_economies.csv')
```

</details>


---


# Congratulations! You have completed the lab.


## Authors


[Abhishek Gagneja](https://www.linkedin.com/in/abhishek-gagneja-23051987/)


## Change Log


|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2023-11-10|0.1|Abhishek Gagneja|Created initial version|


Copyright © 2023 IBM Corporation. All rights reserved.
